# Building a Tropical PyTree Library from CALPHAD Data

This notebook demonstrates how to use the `thermograph` TDB extraction utilities to automatically parse a CALPHAD database (`.tdb`) and compile all of its functional dependencies into highly optimized `ZGraphNode` binaries (`.zg`).

These compiled phases will be saved directly into the centralized `LIBRARY_DIR` and can be loaded in other notebooks using `load_library()`.

In [ ]:
import os
from thermograph.io.tdb import parse_tdb
from zgraph import save
from thermograph.config import LIBRARY_DIR

# Ensure the unary.tdb file exists in the examples directory
tdb_path = 'unary.tdb'

if not os.path.exists(tdb_path):
    print(f"Could not find {tdb_path} in the current directory.")
else:
    tdb_name = os.path.splitext(os.path.basename(tdb_path))[0]
    
    print(f"Extracting all phases from {tdb_path}...")
    # 1. Parse the TDB into a dictionary of ZGraphNodes in memory
    nodes = parse_tdb(tdb_path)
    
    # 2. Serialize the nodes to the binary library directory
    # zgraph.save automatically intercepts the dictionary and writes each node
    # as its own .zg file inside the 'unary' directory.
    save(nodes, tdb_name, library_dir=LIBRARY_DIR)
    
    print(f"Library successfully built in {os.path.join(LIBRARY_DIR, tdb_name)}")

### Testing the Library

Now that the library is compiled, let's test it by loading the Cu FCC phase and plotting its free energy.

In [ ]:
from thermograph import load_library
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Load the compiled TropicalPolynomialNode for Cu FCC
cu_fcc = load_library('unary/GHSERCU')

T_range = jnp.linspace(300, 3000, 500)
G_fcc = jax.vmap(cu_fcc)(jnp.expand_dims(T_range, 1))

plt.figure(figsize=(8, 5))
plt.plot(T_range, G_fcc, label='Cu FCC', color='blue')
plt.xlabel('Temperature (K)')
plt.ylabel('Gibbs Free Energy (J/mol)')
plt.title('Compiled Cu FCC Phase')
plt.legend()
plt.grid(True)
plt.show()